In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel('/content/Seasonally Adjusted.xlsx')

df.head()

In [ ]:
df.shape

In [ ]:
data = pd.read_csv('CPI_MONTHLY (1).csv')
data.head()

In [ ]:
data.rename(columns={'date': 'Date'}, inplace=True)

In [ ]:
# Convert 'Date' column to datetime64[ns] type
data['Date'] = pd.to_datetime(data['Date'])
df['Date'] = pd.to_datetime(df['Date'])

# Assuming both dataframes have a 'Date' column
merged_df = pd.merge(data, df, how='outer', on='Date')

# Drop rows where any value is null
data = merged_df.dropna()

data.head()

In [ ]:
# Assuming df is your original dataframe
df = pd.melt(data, id_vars=['Date'], value_vars=['Single_Family_HPI_SA', 'One_Storey_HPI_SA', 'Two_Storey_HPI_SA', 'Townhouse_HPI_SA', 'Apartment_HPI_SA'], var_name='Type', value_name='Price')

# Rename columns if needed
df.columns = ['Date', 'Type', 'Price']

df.head()

In [ ]:
# Assuming df_long is your reshaped dataframe
df = df.sort_values(by='Date')

# Display the sorted dataframe
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Count plot for the number of entries for each house type
plt.figure(figsize=(10, 5))
sns.countplot(x='Type', data=df)
plt.title('Count of Entries for Each House Type')
plt.xlabel('House Type')
plt.ylabel('Count')

In [ ]:
# Pivot the table to have 'Type' as columns
df_pivot = df.pivot(index='Date', columns='Type', values='Price')

# Fill missing values with 0 (assuming missing values mean no data)
df_pivot.fillna(0, inplace=True)

# Normalize the data
scaler = MinMaxScaler()
df_normalized = scaler.fit_transform(df_pivot)

# Create sequences for time series prediction
sequence_length = 10  # You can adjust this based on your preference
X, y = [], []
for i in range(len(df_normalized) - sequence_length):
    X.append(df_normalized[i:i+sequence_length])
    y.append(df_normalized[i+sequence_length])

X, y = np.array(X), np.array(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the LSTM model
model = Sequential()
model.add(LSTM(units=50, activation='relu', input_shape=(X.shape[1], X.shape[2])))
model.add(Dense(units=df_pivot.shape[1]))  # Assuming one output for each 'Type'
model.compile(optimizer='adam', loss='mse')

model.summary()

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# Plot loss curves
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Make predictions
predictions = model.predict(X_test)

# Inverse transform predictions
predictions_actual = scaler.inverse_transform(predictions)

# Inverse transform y_test to get actual prices
y_test_actual = scaler.inverse_transform(y_test)

# Calculate R2 score
r2 = r2_score(y_test_actual, predictions_actual)
print(f'R2 Score: {r2}')

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test_actual, predictions_actual)
print(f'Mean Squared Error (MSE): {mse}')

# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error (RMSE): {rmse}')

In [ ]:
# Plot actual vs predicted prices for each 'Type'
plt.figure(figsize=(10, 8))
plt.plot(df_pivot.columns, y_test_actual[5], label='Actual', marker='o')
plt.plot(df_pivot.columns, predictions_actual[5], label='Predicted', linestyle='dashed', marker='o')
plt.title(f'Actual vs Predicted Prices for Sample {5 + 1}')
plt.xlabel('Type')
plt.ylabel('Price')
plt.legend()
plt.show()